# Task 2. Interpolation of the internal gaps

Task 2 asks us to fill in the embedded missing blocks inside each series and quantify the uncertainty of the estimates. From Task 1, the most important lessons are:

- raw prices are highly persistent and should not be modeled directly,
- log-returns are much closer to stationary than price levels,
- most series show volatility clustering, so uncertainty should depend on local return variability.

Based on that, the interpolation below works on the log-price scale and compares three models: a random-walk bridge, an ARIMA interpolation, and a GARCH-assisted bridge.


In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from statsmodels.tsa.arima.model import ARIMA

try:
    from arch import arch_model
except ImportError:
    arch_model = None

repo_root = Path.cwd()
if not (repo_root / "config").exists():
    repo_root = repo_root.parent
sys.path.append(str(repo_root))

from config.plot_config import plot_config
from Subtasks.series_analysis import PriceAnalysis
from Subtasks.task2_plots import (
    plot_backtest_example_windows,
    plot_backtest_metric_summary,
    plot_bridge_log_price_construction,
    plot_bridge_uncertainty_profiles,
    plot_true_gap_model_comparison,
)

output = plot_config() / "task2"
output.mkdir(exist_ok=True)


## 1. Load the cleaned data and identify the interpolation targets

We only interpolate the internal missing blocks. The shared trailing 200-day gap belongs to Task 3 and must not be touched here.


In [ ]:
analysis = PriceAnalysis.from_csv("spiff_data-2.csv").clean()
prices = analysis.prices
gap_table = analysis.gap_table()
internal_gaps = gap_table[gap_table["gap_type"] == "internal"].copy()
trailing_gaps = gap_table[gap_table["gap_type"] == "trailing"].copy()

display(internal_gaps)



In [ ]:
fig, ax = plt.subplots()

for column in prices.columns:
    ax.plot(prices.index, prices[column], label=column)

    # gaps for just this series
    gaps = internal_gaps[internal_gaps["series"] == column]

    for _, gap in gaps.iterrows():
        ax.axvspan(
            gap["start"],
            gap["end"],
            alpha=0.15
        )

ax.set_title("Price series with gaps")
ax.set_xlabel("day")
ax.set_ylabel("price")
ax.legend(loc="upper left")
ax.set_xlim(0,1500)
plt.tight_layout()
plt.savefig(output / "cleaned_raw_price_series.png", bbox_inches="tight")
plt.show()

## 2. Model 1: random-walk bridge

This is the baseline model. We work with log-prices, use the observed endpoint before and after the gap, and interpolate the conditional random-walk path between them.

If the gap has length $m$, and the observed log-prices at the two endpoints are $Y_0$ and $Y_{m+1}$, the bridge mean is

$$
\mathbb{E}[Y_h \mid Y_0, Y_{m+1}] = Y_0 + \frac{h}{m+1}(Y_{m+1} - Y_0), \qquad h = 1, \dots, m.
$$

The uncertainty uses a locally estimated return variance:

$$
\operatorname{Var}(Y_h \mid Y_0, Y_{m+1}) = \sigma^2 \frac{h(m+1-h)}{m+1}.
$$

We use this model as the default filled series because it is transparent, stable, and explicitly conditioned on both endpoints.


In [ ]:
log_returns = analysis.log_returns
variance_window = 60
z_value = 1.96
validation_gap_length = int(internal_gaps["length"].mode().iloc[0])
max_validation_windows = 12
context_window = 40


def bridge_interpolate_window(series, return_series, start, end, variance_window=60, z_value=1.96):
    gap_length = end - start + 1
    left_log_price = np.log(series.loc[start - 1])
    right_log_price = np.log(series.loc[end + 1])
    steps = np.arange(1, gap_length + 1)
    bridge_mean = left_log_price + steps / (gap_length + 1) * (right_log_price - left_log_price)

    local_returns = pd.concat(
        [
            return_series.loc[: start - 1].dropna().tail(variance_window),
            return_series.loc[end + 2 :].dropna().head(variance_window),
        ]
    )
    local_variance = local_returns.var(ddof=1)
    if pd.isna(local_variance) or local_variance <= 0:
        local_variance = return_series.dropna().var(ddof=1)

    bridge_variance = local_variance * steps * (gap_length + 1 - steps) / (gap_length + 1)
    bridge_std = np.sqrt(bridge_variance)
    gap_index = pd.Index(range(start, end + 1), name=series.index.name)

    return pd.DataFrame(
        {
            "estimate": np.exp(bridge_mean),
            "lower_95": np.exp(bridge_mean - z_value * bridge_std),
            "upper_95": np.exp(bridge_mean + z_value * bridge_std),
            "log_price_mean": bridge_mean,
            "log_price_std": bridge_std,
            "variance_source": "local_window",
        },
        index=gap_index,
    )




## 3. Model 2: ARIMA interpolation

The ARIMA model is a mean-path alternative. We mask the target gap in the log-price series, fit ARIMA(1,1,1), and use the model-smoothed predictions inside the gap.

This checks whether fitted time-series dynamics improve the point estimates compared with the simple endpoint-conditioned bridge.


In [ ]:
arima_order = (1, 1, 1)


def arima_interpolate_window(series, return_series, start, end, order=(1, 1, 1), z_value=1.96, **_):
    log_series = np.log(series.astype(float)).copy()
    log_series.loc[start:end] = np.nan
    last_observed_day = int(series.last_valid_index())
    model_data = log_series.loc[:last_observed_day]

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model = ARIMA(
            model_data,
            order=order,
            trend="n",
            enforce_stationarity=False,
            enforce_invertibility=False,
            missing="none",
        )
        result = model.fit()
        prediction = result.get_prediction(start=start, end=end)

    gap_index = pd.Index(range(start, end + 1), name=series.index.name)
    log_price_mean = np.asarray(prediction.predicted_mean, dtype=float)
    confidence_interval = prediction.conf_int(alpha=0.05)
    lower_log = np.asarray(confidence_interval.iloc[:, 0], dtype=float)
    upper_log = np.asarray(confidence_interval.iloc[:, 1], dtype=float)
    log_price_std = (upper_log - lower_log) / (2 * z_value)

    return pd.DataFrame(
        {
            "estimate": np.exp(log_price_mean),
            "lower_95": np.exp(lower_log),
            "upper_95": np.exp(upper_log),
            "log_price_mean": log_price_mean,
            "log_price_std": log_price_std,
            "variance_source": f"ARIMA{order}",
        },
        index=gap_index,
    )




## 4. Model 3: GARCH-assisted bridge

The GARCH model keeps the same endpoint-conditioned bridge mean idea, but changes the uncertainty calculation.

Instead of one constant local variance, it fits GARCH(1,1) to pre-gap log-returns and uses step-specific conditional variance forecasts. This tests whether volatility clustering improves the interval shape.


In [ ]:
garch_window = 250


def _variance_bridge_from_step_variances(series, start, end, step_variances, z_value=1.96, variance_source="step_variance"):
    gap_length = end - start + 1
    horizon = gap_length + 1
    if len(step_variances) < horizon:
        raise ValueError("Need one variance forecast for each hidden increment plus the endpoint increment.")

    increment_variance = np.maximum(np.asarray(step_variances[:horizon], dtype=float), 1e-12)
    cumulative_variance = np.cumsum(increment_variance)
    total_variance = cumulative_variance[-1]
    hidden_cumulative_variance = cumulative_variance[:-1]
    weights = hidden_cumulative_variance / total_variance

    left_log_price = np.log(series.loc[start - 1])
    right_log_price = np.log(series.loc[end + 1])
    bridge_mean = left_log_price + weights * (right_log_price - left_log_price)
    bridge_variance = hidden_cumulative_variance * (1 - weights)
    bridge_std = np.sqrt(np.maximum(bridge_variance, 0))
    gap_index = pd.Index(range(start, end + 1), name=series.index.name)

    return pd.DataFrame(
        {
            "estimate": np.exp(bridge_mean),
            "lower_95": np.exp(bridge_mean - z_value * bridge_std),
            "upper_95": np.exp(bridge_mean + z_value * bridge_std),
            "log_price_mean": bridge_mean,
            "log_price_std": bridge_std,
            "variance_source": variance_source,
        },
        index=gap_index,
    )


def garch_bridge_interpolate_window(
    series,
    return_series,
    start,
    end,
    variance_window=250,
    z_value=1.96,
    min_observations=80,
):
    if arch_model is None:
        raise ImportError("Install the `arch` package to run the GARCH candidate.")

    gap_length = end - start + 1
    pre_gap_returns = return_series.loc[: start - 1].dropna().tail(variance_window)
    if len(pre_gap_returns) < min_observations:
        fallback = bridge_interpolate_window(series, return_series, start, end, variance_window=variance_window, z_value=z_value)
        fallback["variance_source"] = "local_window_fallback"
        return fallback

    scaled_returns = pre_gap_returns * 100
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model = arch_model(scaled_returns, mean="Zero", vol="GARCH", p=1, q=1, dist="normal", rescale=False)
        result = model.fit(disp="off", show_warning=False)
        forecast = result.forecast(horizon=gap_length + 1, reindex=False)

    step_variances = forecast.variance.iloc[-1].to_numpy() / 10_000
    if not np.isfinite(step_variances).all() or np.any(step_variances <= 0):
        fallback = bridge_interpolate_window(series, return_series, start, end, variance_window=variance_window, z_value=z_value)
        fallback["variance_source"] = "local_window_fallback"
        return fallback

    return _variance_bridge_from_step_variances(
        series,
        start,
        end,
        step_variances,
        z_value=z_value,
        variance_source="GARCH(1,1)_pre_gap",
    )




## 5. Shared model registry and backtest setup

The functions below collect the available models and evaluate them on the same pseudo-gaps. This keeps the model comparison fair: every model receives the same hidden windows and is scored with the same metrics.


In [ ]:
def candidate_gap_starts(series, gap_length):
    starts = []
    first_start = int(series.index.min()) + 1
    last_start = int(series.index.max()) - gap_length

    for start in range(first_start, last_start + 1):
        end = start + gap_length - 1
        if series.loc[start - 1 : end + 1].notna().all():
            starts.append(start)

    return starts


def build_interpolation_models():
    def arima_model(series, return_series, start, end, **kwargs):
        return arima_interpolate_window(series, return_series, start, end, order=arima_order, **kwargs)

    def garch_model(series, return_series, start, end, **kwargs):
        model_kwargs = {**kwargs, "variance_window": garch_window}
        return garch_bridge_interpolate_window(series, return_series, start, end, **model_kwargs)

    models = {
        "bridge": bridge_interpolate_window,
        "ARIMA(1,1,1)": arima_model,
    }

    if arch_model is not None:
        models["GARCH(1,1)-bridge"] = garch_model
    else:
        print("Skipping GARCH candidate because the `arch` package is not installed. Run `pip install -r requirements.txt` to include it.")

    return models


def evaluate_interpolators(prices, log_returns, interpolators, gap_length, variance_window=60, max_windows=12, z_value=1.96):
    rows = []

    for column in prices.columns:
        series = prices[column]
        return_series = log_returns[column]
        starts = candidate_gap_starts(series, gap_length)

        if len(starts) > max_windows:
            selected = np.linspace(0, len(starts) - 1, max_windows, dtype=int)
            starts = [starts[i] for i in selected]

        for start in starts:
            end = start + gap_length - 1
            true_prices = series.loc[start:end]
            true_log_prices = np.log(true_prices)

            for model_name, interpolator in interpolators.items():
                try:
                    interpolation = interpolator(
                        series,
                        return_series,
                        start,
                        end,
                        variance_window=variance_window,
                        z_value=z_value,
                    )
                    rows.append(
                        {
                            "model": model_name,
                            "series": column,
                            "start": start,
                            "end": end,
                            "price_rmse": float(
                                np.sqrt(np.mean((interpolation["estimate"].to_numpy() - true_prices.to_numpy()) ** 2))
                            ),
                            "log_price_rmse": float(
                                np.sqrt(
                                    np.mean((interpolation["log_price_mean"].to_numpy() - true_log_prices.to_numpy()) ** 2)
                                )
                            ),
                            "coverage_95": float(
                                np.mean(
                                    (true_prices.to_numpy() >= interpolation["lower_95"].to_numpy())
                                    & (true_prices.to_numpy() <= interpolation["upper_95"].to_numpy())
                                )
                            ),
                            "avg_interval_width": float(np.mean(interpolation["upper_95"] - interpolation["lower_95"])),
                            "model_note": interpolation.get("variance_source", pd.Series([""])).iloc[0],
                        }
                    )
                except Exception as error:
                    rows.append(
                        {
                            "model": model_name,
                            "series": column,
                            "start": start,
                            "end": end,
                            "price_rmse": np.nan,
                            "log_price_rmse": np.nan,
                            "coverage_95": np.nan,
                            "avg_interval_width": np.nan,
                            "model_note": repr(error),
                        }
                    )

    return pd.DataFrame(rows)


interpolation_models = build_interpolation_models()


## 6. Backtest the models with pseudo-gaps

To evaluate expected performance without leaking information from the true missing block, we create artificial 50-day gaps in fully observed stretches of each series. Each model receives the same hidden windows, and we compare both point accuracy and interval calibration. The plots below show both aggregate model metrics and selected pseudo-gap reconstructions.


In [ ]:
backtest_results = evaluate_interpolators(
    prices,
    log_returns,
    interpolation_models,
    gap_length=validation_gap_length,
    variance_window=variance_window,
    max_windows=max_validation_windows,
    z_value=z_value,
)

metrics = ["price_rmse", "log_price_rmse", "coverage_95", "avg_interval_width"]
backtest_summary = (
    backtest_results.groupby(["model", "series"])[metrics]
    .mean()
    .sort_values(["model", "log_price_rmse"])
)
overall_backtest = backtest_results.groupby("model")[metrics].mean().sort_values("log_price_rmse")

display(overall_backtest)
display(backtest_summary)

needs_attention = backtest_results[metrics].isna().any(axis=1) | backtest_results["model_note"].astype(str).str.contains(
    "fallback|Error|Exception|ImportError", regex=True
)
model_notes = backtest_results.loc[needs_attention, ["model", "series", "start", "model_note"]]
if not model_notes.empty:
    display(model_notes.drop_duplicates().head(20))

plot_backtest_metric_summary(overall_backtest, output / "backtest_model_metrics.png")
plot_backtest_example_windows(
    prices,
    log_returns,
    backtest_results,
    interpolation_models,
    output / "backtest_example_windows.png",
    gap_length=validation_gap_length,
    variance_window=variance_window,
    z_value=z_value,
    context_window=context_window,
)


The overall table is the main model check. Lower `log_price_rmse` means better point reconstruction on the log scale, while `coverage_95` near 0.95 means the uncertainty bands are reasonably calibrated. ARIMA is a direct mean-model alternative; the GARCH bridge mainly tests whether volatility clustering improves the interval shape.


## 7. Fill the true gaps with the random-walk bridge

For the real missing blocks, using both endpoints of the gap is legitimate because the task is interpolation rather than forecasting. The random-walk bridge remains the default filled series because it is transparent and endpoint-conditioned. We plot the bridge on the log-price scale first, then compare ARIMA and GARCH alternatives in the next section.


In [ ]:
interpolated_prices = prices.copy()
lower_95_prices = prices.copy()
upper_95_prices = prices.copy()
interpolated_gap_frames = []
interpolation_summary_rows = []

for row in internal_gaps.itertuples(index=False):
    gap_interpolation = analysis.interpolate_internal_gap(row.series, variance_window=variance_window, z_value=z_value)
    interpolated_gap_frames.append(gap_interpolation.reset_index().rename(columns={prices.index.name: "day"}))

    interpolated_prices.loc[gap_interpolation.index, row.series] = gap_interpolation["estimate"].to_numpy()
    lower_95_prices.loc[gap_interpolation.index, row.series] = gap_interpolation["lower_95"].to_numpy()
    upper_95_prices.loc[gap_interpolation.index, row.series] = gap_interpolation["upper_95"].to_numpy()

    midpoint_day = int(gap_interpolation.index[len(gap_interpolation) // 2])
    interpolation_summary_rows.append(
        {
            "series": row.series,
            "gap_start": row.start,
            "gap_end": row.end,
            "gap_length": row.length,
            "midpoint_day": midpoint_day,
            "midpoint_estimate": float(gap_interpolation.loc[midpoint_day, "estimate"]),
            "avg_interval_width": float(np.mean(gap_interpolation["upper_95"] - gap_interpolation["lower_95"])),
            "local_return_variance": float(gap_interpolation["local_return_variance"].iloc[0]),
        }
    )

interpolated_gap_values = pd.concat(interpolated_gap_frames, ignore_index=True).set_index(["series", "day"])
interpolation_summary = pd.DataFrame(interpolation_summary_rows).set_index("series").sort_index()

display(interpolation_summary)
display(interpolated_gap_values.head(15))

plot_bridge_log_price_construction(
    prices,
    interpolated_gap_values,
    internal_gaps,
    output / "bridge_log_price_construction.png",
    context_window=context_window,
    z_value=z_value,
)
plot_bridge_uncertainty_profiles(
    interpolated_gap_values,
    internal_gaps,
    output / "bridge_uncertainty_profiles.png",
)


## 8. Compare model estimates on the true gaps

The table below does not overwrite the bridge-filled series. It reports the midpoint and average interval width from each model, and the following plot overlays the competing estimates inside each actual internal gap.


In [ ]:
true_gap_model_frames = []
true_gap_model_summary_rows = []

for row in internal_gaps.itertuples(index=False):
    series = prices[row.series]
    return_series = log_returns[row.series]

    for model_name, interpolator in interpolation_models.items():
        try:
            model_interpolation = interpolator(
                series,
                return_series,
                row.start,
                row.end,
                variance_window=variance_window,
                z_value=z_value,
            )
        except Exception as error:
            true_gap_model_summary_rows.append(
                {
                    "model": model_name,
                    "series": row.series,
                    "midpoint_day": np.nan,
                    "midpoint_estimate": np.nan,
                    "avg_interval_width": np.nan,
                    "model_note": repr(error),
                }
            )
            continue

        model_frame = model_interpolation.reset_index().rename(columns={prices.index.name: "day"})
        model_frame.insert(0, "series", row.series)
        model_frame.insert(0, "model", model_name)
        true_gap_model_frames.append(model_frame)

        midpoint_day = int(model_interpolation.index[len(model_interpolation) // 2])
        true_gap_model_summary_rows.append(
            {
                "model": model_name,
                "series": row.series,
                "midpoint_day": midpoint_day,
                "midpoint_estimate": float(model_interpolation.loc[midpoint_day, "estimate"]),
                "avg_interval_width": float(np.mean(model_interpolation["upper_95"] - model_interpolation["lower_95"])),
                "model_note": model_interpolation.get("variance_source", pd.Series([""])).iloc[0],
            }
        )

true_gap_model_summary = pd.DataFrame(true_gap_model_summary_rows).set_index(["model", "series"]).sort_index()
true_gap_model_values = (
    pd.concat(true_gap_model_frames, ignore_index=True).set_index(["model", "series", "day"])
    if true_gap_model_frames
    else pd.DataFrame()
)

display(true_gap_model_summary)
display(true_gap_model_summary["midpoint_estimate"].unstack("model"))

plot_true_gap_model_comparison(
    prices,
    true_gap_model_values,
    internal_gaps,
    output / "true_gap_model_comparison.png",
    context_window=context_window,
)


In [ ]:
fig, axes = plt.subplots(len(internal_gaps), 1, figsize=(12, 2.8 * len(internal_gaps)), squeeze=False)

for row_idx, gap in enumerate(internal_gaps.itertuples(index=False)):
    ax = axes[row_idx, 0]
    left = max(int(prices.index.min()), gap.start - context_window)
    right = min(int(prices.index.max()), gap.end + context_window)
    window_index = range(left, right + 1)
    gap_values = interpolated_gap_values.xs(gap.series, level="series").loc[gap.start : gap.end]

    ax.plot(prices.loc[window_index, gap.series].index, prices.loc[window_index, gap.series], color="0.55", label="observed price")
    ax.plot(
        gap_values.index,
        gap_values["estimate"],
        color="tab:blue",
        marker="o",
        markersize=2.8,
        linewidth=1.8,
        label="interpolated_gap_values estimate" if row_idx == 0 else None,
    )
    ax.fill_between(
        gap_values.index,
        gap_values["lower_95"].to_numpy(),
        gap_values["upper_95"].to_numpy(),
        color="tab:blue",
        alpha=0.2,
        label="95% interval" if row_idx == 0 else None,
    )
    ax.axvspan(gap.start, gap.end, color="tab:orange", alpha=0.08)
    ax.set_title(f"{gap.series}: interpolation of days {gap.start}-{gap.end}")
    ax.set_ylabel("price")

axes[0, 0].legend(loc="upper left")
axes[-1, 0].set_xlabel("day")
plt.tight_layout()
plt.savefig(output / "interpolated_gap_windows.png", bbox_inches="tight")
plt.show()


## 9. Conclusions and expected performance

This gives one baseline fill model plus two explicit checks.

Main points:

- The random-walk bridge respects the lessons from Task 1 by interpolating on the log-price scale and conditioning on both observed endpoints.
- ARIMA(1,1,1) tests whether a fitted log-price dynamics model can reconstruct masked gaps better than the bridge.
- The GARCH(1,1)-assisted bridge tests whether volatility clustering improves the uncertainty bands by using forecast conditional variances instead of one constant local variance.
- The pseudo-gap backtest is the decision rule: prefer a more complex model only if it improves log-price RMSE without damaging 95% coverage.
- The actual missing values are stored in `interpolated_prices` using the random-walk bridge baseline, while `true_gap_model_summary` and `true_gap_model_values` contain the ARIMA/GARCH alternatives for comparison.

If the GARCH row is absent, install the dependencies in `requirements.txt` and rerun the notebook so the optional `arch` package is available.
